# NB26 — LTR with graded-relevance labels vs binary click labels

**Regime:** `[LAB]` four-cell labels (requires NB22 gaze-regression), cursor-deployable at inference-time features.

*(Authoritative Key Claims block lives in the next cell — regenerated by `notebooks-v2/update_key_claims.py`.)*

## Context: LTR design

The graded-relevance reframe (CIKM §4.2, §4.3) replaces framing deferred items as hard negatives. This notebook is the **empirical validation** of that reframe: does training an LTR ranker on the 3-cell graded labels *actually* improve downstream MRR@10 over training on binary click labels, on the same feature set and same held-out participants?

**Label scheme:**

- Clicked → grade 2
- NB22 deferred → grade 1
- Eval-rejected AND not-approached *above* click position → grade 0
- **Exclude** not-approached records *below* click position. Not evaluated, not rejected — structurally unseen. Training on them as 0 teaches a confound.

**Feature set (5 features, no position):**

1. `lexical_overlap(query_tokens, title+snippet_tokens)` — fraction of query tokens appearing in result text
2. `avg_term_frequency(query_tokens, title+snippet_tokens)` — mean count per query term
3. `cos_sim(query, title)`
4. `cos_sim(query, snippet)`
5. `cos_sim(query, title+snippet)` — cached in `serp-embeddings.json`

**Position is intentionally excluded** as a feature. The argument: position-bias orthodoxy is a trick that assumes position confounds the label; in forced-choice AdSERP where every item is fixated, there is no confound to correct for.

**CV protocol:** leave-one-participant-out, 47 folds (matches §4.1 M4 LOSO).

**Ranker:** sklearn LogisticRegression (simpler than LambdaMART/lightgbm; scores each record pointwise; sorted by score to produce a per-trial ranking). The design works for LR or LambdaMART — LR is sufficient for the binary-vs-graded contrast.

**Headline:** does MRR(graded) > MRR(binary) on the paired per-participant delta? If yes, the graded cell labels add downstream ranker-training information that binary click labels miss — empirical validation of the graded-relevance reframe.

## Key Claims (authoritative for paper writers)

*Last verified against executed notebook output: 2026-04-12.*
*Notebook: `26_ltr_graded_relevance.ipynb`.*

If prose in a paper draft cites a value that disagrees with a row below, the paper is wrong — not the notebook. If re-running this notebook produces different values, update this block immediately and `grep` for the old value across `docs/`.

### Original null-findings protocol (2026-04-15) — labeled-subset MRR, LR/Ridge, 5 text features

**Regime:** `[LAB]` four-cell labels (requires NB22 gaze-regression). 47-fold LOPO. Training-side exclusion: not-approached-below-click records dropped.

| ID | Claim | Value |
|---|---|---|
| **K1** | MRR@labeled-subset on original SERP ordering (Google baseline) | **0.4114** |
| **K2** | MRR@labeled-subset on LR binary ranker | **0.6108** |
| **K3** | MRR@labeled-subset on Ridge graded ranker | **0.6152** |
| **K4** | Paired Δ(graded − binary) on labeled subset, 47-participant Wilcoxon one-sided | Δ = **+0.0046 ± 0.0209**, 31/47, *W* = 720.5, ***p* = 0.0246** |
| **K5** | Paired Δ(graded − original) on labeled subset, 47-participant Wilcoxon one-sided | Δ = **+0.2065**, *W* = 1128, ***p* ≈ 1 × 10⁻⁹** |

The labeled-subset framing is known to compress the MRR competition — the training-side exclusion drops rarely-clicked positions 4–9 where Google trivially wins, artificially narrowing the ranker-vs-Google gap. See `docs/null-findings/nb26-ltr-graded-vs-binary.md` for the full-SERP reanalysis that revealed the original headline was labeled-subset-artefact-inflated.

### 2026-04-19 extension — full-SERP MRR, LambdaMART, M4 cursor features

*Extension rows K6–K16 verified against executed notebook output on 2026-04-19 (notebook cell 15). The file-wide `VERIFIED` date above covers the prior transcription pass for the other notebooks.*

**Regime:** `[LAB]`. 47-fold LOPO. Training uses the not-approached-below-click exclusion; held-out **inference scores all 10 positions per trial** (stricter than the null-doc "scorable-subset" protocol). LGBM rungs are averaged across seeds 0/1/2 for stability. 1,826 held-out trials (trials with missing embeddings at any position are dropped — stricter than the null-doc protocol).

| ID | Claim | Value |
|---|---|---|
| **K6** | Full-SERP MRR on original SERP ordering (Google baseline) | **0.4125** |
| **K7** | Full-SERP MRR, Rung 0: LR binary on 5 text features | **0.2878** |
| **K8** | Full-SERP MRR, Rung 0: Ridge graded on 5 text features | **0.2922** |
| **K9** | Full-SERP MRR, Rung 1: LambdaMART binary on 5 text features | **0.2893** |
| **K10** | Full-SERP MRR, Rung 1: LambdaMART graded on 5 text features | **0.2821** |
| **K11** | Full-SERP MRR, Rung 2: LambdaMART binary on 5 text + 9 M4 features (leakage) | **0.3723** |
| **K12** | Full-SERP MRR, Rung 2: LambdaMART graded on 5 text + 9 M4 features (leakage) | **0.4326** |

Paired per-participant Wilcoxon signed-rank (one-sided, 47 participants):

| ID | Claim | Value |
|---|---|---|
| **K13** | Rung 1: graded − binary paired Δ (LambdaMART on text) | Δ = **−0.0042 ± 0.0450**, 22/47, *p* = **0.7891** (ns) |
| **K14** | Rung 2: graded − binary paired Δ (LambdaMART on text + M4) | Δ = **+0.0591 ± 0.0826**, **39/47**, ***p* < 0.0001** |
| **K15** | Ranker-family isolated: Rung 1 graded − Rung 0 Ridge graded | Δ = **−0.0041 ± 0.0537**, 23/47, *p* = **0.6645** (ns) |
| **K16** | Feature-add isolated: Rung 2 graded − Rung 1 graded | Δ = **+0.1343 ± 0.1458**, 39/47, ***p* < 0.0001** |

**Notable negatives** (defensibly bound the story):

- R2 LGBM graded − Original (Google): Δ = +0.0079 ± 0.1354, 21/47, *p* = 0.4687 (ns). Even with M4 leakage, Rung 2 graded does not significantly beat Google on paired per-participant MRR. The leakage is not enough to produce a deployable-ranker claim.
- R2 LGBM binary − Original: Δ = −0.0512, 17/47, *p* = 0.9962 (ns). Rung 2 binary loses to Google.
- R0 Ridge graded − R0 LR binary: Δ = +0.0039, 29/47, *p* = 0.0719 (full-SERP analogue of K4; marginal under stricter eval).

**M4 leakage caveat — reportable caveat on K11, K12, K14, K16.** The 9 M4 cursor features are aggregates over the full cursor trajectory on each result. For clicked results the trajectory includes the movement *to* the click target, so clicked records carry click-indicative feature values: `min_dist` ≈ 73 px median vs 235 px for not-clicked; `dwell_in_proximity_ms` ≈ 1760 ms vs 194 ms. This means the Rung-2 comparisons probe a regime where the features partially encode the click.

The K14 graded-vs-binary Δ holds *feature distribution and LOPO splits constant* across the contrast; what it does **not** hold constant is loss geometry — LambdaMART with `label_gain=[0,1]` optimizes a different pairwise ranking loss than with `label_gain=[0,1,2]`, and the graded loss can exploit the leaky features more aggressively to separate clicked (gain 2) from deferred (gain 1). The defensible reading of K14 is therefore: **isolates label encoding plus any loss-structure interaction with the leaky features**, not label encoding alone. A pre-click-truncated M4 variant (not yet built) is what would attribute the +0.0591 cleanly to the graded labels. The K11/K12 absolute MRRs are not deployable-ranker values and should not be cited as such.

**What the extension shows about the null:**

- **Ranker family alone does not break the null (K15).** LambdaMART with 5 text features does not meaningfully outperform Ridge on the same features (paired Δ = −0.0041, ns). Minor caveat: K15 compares seed-averaged LGBM to deterministic Ridge, which biases the comparison *toward* finding no effect — the null is conservatively established.
- **Feature addition yields a paired graded-vs-binary lift at Rung 2 (K14, K16) — with the loss-geometry + leakage caveat above.** +0.0591 (p < 0.0001, 39/47) for graded-vs-binary at Rung 2; +0.1343 (p < 0.0001, 39/47) for feature-add isolated.
- **No leakage-driven "beat Google" headline.** R2 graded − Original is not significant (+0.0079, p = 0.47). Note this bounds *deployability*, not the contribution of leakage to the K14 contrast — those are separate claims.

**WILD gate — deferred.** ACD has one AOI per session and no SERP HTML, so a graded-label LTR replication is blocked. A binary ad-click LTR on ACD would only probe ranker-family effects, but K15 already indicates no lift on LAB text features — no pending WILD signal today. Revisit once a validated cursor-only deferred proxy exists (blocked per `attentional-foraging/CLAUDE.md`).

### 2026-04-19/20 extension — viewport features (Rung 2b / Rung 3) and `withstood_evaluation` continuous labels (Rung 4)

*Values for K17–K26 come from executed script output, not notebook cells: `scripts/output/nb26_viewport_rungs/summary.json` (2026-04-19, Rungs 2b/3), `scripts/output/nb26_rung4_withstood/summary.json` (2026-04-19, Rung 4a–4c), and `scripts/output/nb26_rung4_variants/summary.json` (2026-04-20, Rung 4e–4g). Producer scripts: `scripts/nb26_viewport_rungs.py`, `scripts/nb26_rung4_withstood.py`, `scripts/nb26_rung4_variants.py`. The trial pool for this extension is **2,115 full-SERP trials** (stricter filter than the K6–K16 1,826-trial set: requires (trial, pos) presence in both `cursor-approach-features.json` and the new `viewport-trajectory-features.json`). K17 re-reports the Google baseline on this new pool for apples-to-apples paired comparison.*

**Regime:** `[LAB]`. 47-fold LOPO, seeds 0/1/2 averaged. `R1` = 5 text features; `R2a` = text + 9 M4 cursor (leakage); `R2b` = text + 6 viewport/trajectory (cursor-free); `R3` = text + M4 + VP (kitchen sink, M4 leakage carried through). All paired Wilcoxon signed-rank one-sided (greater), per-participant aggregation.

| ID | Claim | Value |
|---|---|---|
| **K17** | Full-SERP MRR on original SERP ordering (Google baseline, 2,115-trial extension pool) | **0.4924** |
| **K18** | Full-SERP MRR, Rung 2b: LambdaMART 3-grade graded on text + VP (cursor-free) | **0.2634** (NDCG@10 = 0.4320) |
| **K19** | Full-SERP MRR, Rung 3: LambdaMART 3-grade graded on text + M4 + VP | **0.6770** (NDCG@10 = 0.7588) |
| **K20** | Feature-add isolated: R3 graded − R2a graded (VP on top of M4 cursor) | Δ = **−0.0994 ± 0.1192**, 4/47, *p* = 1.0000 (ns; **VP degrades with M4**) |
| **K21** | Feature-add isolated: R3 graded − R2b graded (M4 on top of VP) | Δ = **+0.4230 ± 0.1660**, **47/47**, ***p* = 7.1 × 10⁻¹⁵** |

**Viewport-alone null (K18, K20) — writeup:** `docs/null-findings/nb29-viewport-bands-content-residualization.md` and the bg-run log at `scripts/output/nb26_viewport_rungs/run.log`. Viewport + trajectory features on their own are a weak ranking signal (K18 MRR 0.26 < text-alone R1 graded 0.28) and do not add on top of M4 cursor features (K20 paired Δ = −0.099, ns). The viewport signal is real for the deferred-vs-rejected classification (NB28:K4, NB30:K13) but does not transfer to full-SERP ranking when M4 cursor features are already in the model. Probable mechanism: M4 cursor features already absorb the AOI-attention signal that viewport visibility encodes; VP adds variance without additional per-click discrimination.

### Rung 4 — `withstood_evaluation` continuous labels

The Rung 4 extension replaces the 3-grade `{clicked, deferred, eval-rejected/not-approached}` label with variants of a 10-grade continuous relevance derived from the `withstood_evaluation` composite (see `scripts/build_withstood_evaluation_score.py` — a per-(trial, pos) z-score composite of `n_reversals`, `−min_abs_velocity`, `vt_center_fraction`, `dwell_in_proximity_ms`). All Rung 4 runs use the R3 feature set (text + M4 + VP) for apples-to-apples comparison with K19.

| ID | Claim | Value |
|---|---|---|
| **K22** | Rung 4a: LambdaMART 10-grade graded, label = rank-within-trial of `withstood_pre_click` (exp gain `[2^i]`) | **MRR = 0.6559** (NDCG@10 = 0.7410) |
| **K23** | Rung 4c: Ridge MSE pointwise on continuous `withstood_pre_click` | **MRR = 0.6885** (NDCG@10 = 0.7656) |
| **K24** | Rung 4f: LambdaMART 10-grade graded, label = **hybrid click-pinned** (grade 9 = clicked; remaining 9 positions ranked 0–8 by `withstood_pre_click`), exp gain `[2^i]` | **MRR = 0.8248** (NDCG@10 = 0.8686) |
| **K25** | Paired Δ(K24 − K19): R4f hybrid 10-grade − R3 3-grade on the same features — **the denser-label headline** | Δ = **+0.1425 ± 0.1000**, **44/47**, ***p* = 9.9 × 10⁻¹⁴** |
| **K26** | Paired Δ(K22 − K19): R4a naive rank-within-trial 10-grade − R3 3-grade — **the null the hybrid rescued** | Δ = **−0.0261 ± 0.1000**, 17/47, *p* = **0.9773** (ns, denser label hurts) |

**Mechanism: why naive rank-within-trial fails (K26), why hybrid succeeds (K25).** On the 2,115 trials, 47 (2.2 %) have their clicked item at the lowest within-trial withstood rank (bucket 0 of 10), because `withstood_pre_click` is a relevance proxy that correlates but does not co-vary with the click. LambdaMART on `g10_pre` labels trains to rank those 47 clicks at the bottom, which zeroes their MRR contribution. The hybrid label pins the click to grade 9 (preserving ground truth) and uses `withstood_pre_click` only to order the 9 non-clicked positions among themselves — this recovers the denser-label pairwise-gradient benefit (≈ 45 informative pairs per trial vs ≤ 3 for 3-grade) without the click-boundary noise. Clicked-item distribution across `g10_pre` buckets: [47, 12, 19, 20, 63, 79, 138, 224, 567, 946] — 71.5 % of clicks at top 2 within-trial ranks, validating `withstood_evaluation` as a strong relevance proxy.

**Ancillary K-rows:**

- **R4b** (10-grade on `withstood_full`, leakage upper bound): MRR 0.6549 — matches R4a (Δ = −0.001), i.e. the pre-click viewport truncation doesn't remove much leakage because cursor `dwell_in_proximity_ms` was not truncated. Leakage magnitude is not the story.
- **R4e** (10-grade `g10_pre`, **linear** gain `[0..9]`): MRR 0.6552 — Δ vs R4a (exp gain) = −0.001, *p* = 0.70 (ns). Gradient-shape hypothesis falsified: the issue is label noise, not gain curvature.
- **R4g** (hybrid 10-grade, linear gain): MRR 0.7912. Δ vs K19 R3 3-grade = +0.1099 (***p* = 3.2 × 10⁻¹²**). Δ vs K24 R4f hybrid exp = −0.0336 — exp gain wins when labels are clean near the top.

Full Rung 4 null writeup: `docs/null-findings/2026-04-20-rung4-rank-within-trial.md`.

**What the Rung 4 extension shows:**

- **Denser supervision beats 3-grade when the label correctly encodes the click (K24, K25).** The hypothesis Andy proposed — "pairwise ranking gives finer gradient for the reranker when labels span 10 grades" — is vindicated at +0.143 MRR (*p* < 10⁻¹³) once the click ground-truth is preserved.
- **The naive operationalisation fails and the reason is diagnostic (K26).** Rank-within-trial of any click-correlated-but-not-click-equivalent scalar will mis-label 1–3 % of clicks as bottom-bucket; denser supervision amplifies rather than absorbs this noise.
- **Ridge MSE pointwise is a dark-horse baseline (K23).** It matches LambdaMART 3-grade (Δ vs R3 = +0.0080, *p* = 0.29 ns) without pairwise ranking, which is relevant for any deployment that can't carry LightGBM.

**Caveats on Rung 4:**

- **Leakage:** `withstood_pre_click` truncates viewport components at click_t but does NOT truncate cursor `dwell_in_proximity_ms` (requires raw cursor-timeline replay, deferred). The R4a-vs-R4b near-equivalence (Δ = −0.001) suggests cursor leakage in the label is not doing heavy lifting, but a cleaner pre-click label would strengthen K24/K25 further.
- **M4 feature leakage still present:** The R3 feature set includes M4 cursor features computed over the full trial, carrying the same click-correlation caveat flagged for K11, K12, K14, K16. K24 is therefore not a deployable-ranker number; it demonstrates the *label-encoding gain*, not the deployable MRR.
- **Trial-pool mismatch:** K17–K26 use the 2,115-trial pool (stricter on VP feature availability) vs K6–K16's 1,826-trial pool. Paired comparisons within this extension (K20, K21, K25, K26) are apples-to-apples; paired comparisons with K6–K16 across pools are not directly comparable.

### Modeling close (2026-04-20)

The Rung 4 extension closes NB26's modeling thread at K24/K25 as the canonical LAB-regime LTR result on AdSERP. Two residual questions were considered and closed without further runs:

1. **True pre-click cursor-dwell truncation** (would recompute `withstood_pre_click` components that currently use full-trial cursor timeline). **Moot for K24/K25** — the hybrid label pins clicked items at grade 9 regardless of their `withstood_pre_click` value, so cursor-dwell leakage in the *label* cannot shift the headline. Confirmed empirically by R4a vs R4b (Δ = −0.001, ns). Feature-side M4 leakage is a separate, known caveat that applies equally to K11–K16 and K24–K25; it's engineering work (raw cursor-timeline replay) deferred as a future pass.
2. **Spread-top labels** (multiple high-withstood items share grade 9 alongside the click). Expected-null by theory: MRR ties non-clicks with the click and degrades the headline metric; NDCG@10 headroom is already small (0.131 remaining at K24) and dominated by the 2.2 % bucket-0-click irreducible noise, not by the label's top-bucket sparsity. No empirical run planned.

Residual headroom (MRR 0.82 → theoretical 1.00) is feature-limited, not label-limited. Further LTR gains on this dataset require either (a) resolving the feature-side M4 leakage with true pre-click trajectory recomputation, or (b) adding pupillometric features (NB05 LHIPA, NB14 LF/HF) as per-(trial, pos) rankers. Both are scope for a successor notebook, not NB26.


### 2026-05-05 extension — Typed cascade migration + Peter's no-position four-class spec

*Values for K27 come from executed script output: `scripts/output/ltr_typed_four_class/summary.json` (2026-05-05). Producer: `scripts/ltr_typed_four_class.py`. Companion baseline: `scripts/output/aoi-consumer-cascade/lambdamart_baseline_typed.json` (`scripts/lambdamart_baseline_organic.py`, dual with-position / no-position conditions). Companion null: `scripts/output/aoi-consumer-cascade/lambdamart_continuous_gain_typed.json` (32-grade p\_click distillation does not beat binary).*

**Regime:** `[LAB, AdSERP, typed]`. 47-fold LOSO. Trial pool: **2,539 trials** with ≥2 positions and ≥1 click on the typed-cascade dataset (`cursor-approach-features-typed.json`, all 9 etypes, 19,774 records). NotApprBelow excluded from training (5,489 records, 27.8 %); inference scores all positions per trial.

**Feature set (M3-no-position):** `total_dwell_ms` + 9 M4 cursor features. **No text features, no viewport features, no position.** This is the strictest Peter-spec feature set and is *not* directly comparable to K17–K26 (which used 2,115-trial pool + text + M4 + VP). The point of K27 is to isolate the **label-encoding gain** under the typed cascade with cursor-only features.

**Spec source:** Peter Dixon-Moses 2026-05-04 (LightGBM LambdaRank, NDCG@10 optimize, MRR@10 evaluate, no position feature, four-class label generator from the AOI taxonomy with NotApprBelow exclusion).

| ID | Claim | Value |
|---|---|---|
| **K27.0** | Original SERP position (no ML, typed pool 2,539) | NDCG@10 = **0.5774**, MRR@10 = **0.4589** |
| **K27.1** | LR pointwise on binary click (M3-no-position, typed) | NDCG@10 = **0.8176**, MRR@10 = **0.7669** |
| **K27.2** | LambdaMART on binary click (M3-no-position, typed, NotApprBelow excluded) | NDCG@10 = **0.7940**, MRR@10 = **0.7409** |
| **K27.3** | **LambdaMART on 4-class graded** (M3-no-position, typed, NotApprBelow excluded) | NDCG@10 = **0.8183**, MRR@10 = **0.7713** |
| **K27.4** | ΔMRR@10 (4-class − binary LambdaMART, same training set) | **+0.0304** |
| **K27.5** | ΔMRR@10 (4-class − Original SERP position) | **+0.3124** |

**What K27 shows:**

- **The four-class graded label encoding adds signal over binary click on the same NotApprBelow-excluded training set (K27.4: ΔMRR@10 = +0.030).** Same features, same folds, same training rows — only the label differs. This is the cleanest version of the graded-vs-binary contrast in the notebook so far.
- **NotApprBelow exclusion costs binary LambdaMART ≈0.058 MRR** on M3-no-position features (the K27.2 binary 0.741 vs the dual-condition baseline's 0.799 with full data — `lambdamart_baseline_typed.json`). The four-class label recovers ≈0.030 of that cost.
- **NDCG@10 favors 4-class more than MRR@10 does** (ΔNDCG = +0.024 vs ΔMRR = +0.030). NDCG rewards graded ordering of non-clicks; MRR rewards only the clicked item's rank. The 4-class label encodes the deferred/eval-rejected gradient that NDCG can use and binary cannot.
- **Pure motor-only LTR is competitive without text or viewport.** K27.3 MRR@10 = 0.771 with M3-no-position cursor features alone is within 0.054 of K24's 10-grade hybrid result on the richer text + M4 + VP feature set. Cursor signal carries most of the relevance information.

**Companion null — continuous p_click distillation does not beat binary on typed (`lambdamart_continuous_gain_typed.json`):**

- Pointwise LR (M3-no-position): MRR = 0.776
- LambdaMART (binary click, full-data): MRR = 0.799
- LambdaMART (32-grade p_click discretization, full-data): MRR = 0.777

The teacher-LR continuous-gain story (flavor-2 of the original click-probability-as-label proposal) is now confirmed dead under the typed cascade. The label gradation has to come from a **behavioral source** (the four-class taxonomy), not from a probability-distillation derivative.

**Caveats on K27:**

- **Pool change (2,115 → 2,539 trials)**: K17–K26 require viewport-trajectory feature presence; K27 only requires typed-cascade record presence. K27's pool is therefore larger but covers a slightly different trial subset. Not directly comparable to K17–K26.
- **Feature set is M3-no-position only.** K27 deliberately excludes text features and viewport features to isolate the cursor-motor signal. A `K28` extension comparing 4-class graded across {M3-no-pos, +text, +VP, +pupillometric} feature stacks is the natural successor (see "Modeling close" section below for the open question on pupillometric LF/HF as a graded-relevance feature).
- **NotApprBelow exclusion is a training-side constraint, not an inference-side one.** Held-out scoring covers all positions per trial; only the training rows are restricted. This means the headline MRR/NDCG values are full-SERP eval, not labeled-subset eval.
- **No ad/non-ad cohort split yet.** The typed cascade carries `etype` per record; per-etype K27 metrics are the natural follow-up (would test whether the four-class signal is etype-invariant or organic-specific, paralleling `four_class_taxonomy_hybrid.py`'s motor-signature dissociation).

**Migration footnote:** the original `scripts/ltr_graded_vs_click.py` (Apr 15) is retired with a deprecation banner. It used pre-typed-cascade input (`cursor-approach-features.json`, Apr 12, also pre-prefix-bug-fix), sklearn pointwise GBR (not LightGBM LambdaRank), and treated NotApprAbove and NotApprBelow identically. K27 supersedes it.


### 2026-05-05 extension — K28/K29: pupillary LF/HF as feature and as pairwise-preference label

*Values from executed script output: `scripts/output/ltr_typed_lfhf_feature/summary.json` (K28) and `scripts/output/ltr_typed_lfhf_pairwise/summary.json` (K29). Producers: `scripts/ltr_typed_lfhf_feature.py`, `scripts/ltr_typed_lfhf_pairwise.py`. LF/HF source: `AdSERP/data/butterworth-lfhf-by-position-typed.json` (per-(trial, pos) LF/HF computed by `scripts/compute_butterworth_lfhf.py`, 28.7 % non-null coverage on the typed-cascade record set).*

**Regime:** `[LAB, AdSERP, typed]`. 47-fold LOSO. Same 2,539-trial typed pool as K27. Same M3-no-position feature set. Only the label / feature changes from K27.3.

**Two questions:**
- **K28** — does adding `lfhf` (and `lfhf_n_samples`) as features improve the K27.3 4-class graded LambdaMART?
- **K29** — does using LF/HF as a *label generator* (rank-within-trial — Andy's "pairwise preferences" framing) improve over the 4-class behavioral label?

| ID | Claim | Value |
|---|---|---|
| **K28.1** | LambdaMART 4-class graded on M3-no-position + lfhf, vs K27.3 (M3-no-position only) | NDCG@10 = **0.8187** (Δ **+0.0004**), MRR@10 = **0.7699** (Δ **−0.0014**) |
| **K28.2** | LambdaMART binary click on M3-no-position + lfhf, vs M3-no-position-only baseline | MRR@10 = **0.7243** (Δ **−0.0167**) |

**K28 verdict (label-side null):** LF/HF as a per-(trial, pos) feature **does not help LTR** when M3 cursor features are already in the model. Binary LambdaMART actively gets worse (Δ −0.017). Mechanism: cursor `total_dwell_ms` and `dwell_in_proximity_ms` already encode the engagement-correlated signal that LF/HF captures, and LF/HF adds variance from its 71 % missing-value mass.

| ID | Claim | Value |
|---|---|---|
| **K29a** | LambdaMART on **pure LF/HF rank-within-trial** label (10-grade, ≥3 non-null lfhf per trial — 514 trials trained, 2,539 evaluated) | NDCG@10 = **0.5263** (Δ **−0.2920**), MRR@10 = **0.4174** (Δ **−0.3539**) |
| **K29b** | LambdaMART on **hybrid 4-class × LF/HF composite** label (NotApprBelow excluded, LF/HF as within-bucket tiebreaker, null lfhf → bucket-bottom — 11,101 rows / 2,416 trials trained) | NDCG@10 = **0.8295** (Δ **+0.0112**), MRR@10 = **0.7854** (Δ **+0.0140**) |

**K29 verdict — pairwise-preferences vision validated for the hybrid design:**

- **K29a fails clearly.** Pupil alone does not order positions well enough to recover clicks (ΔMRR@10 = −0.354 vs the 4-class baseline). Only 514 trials have enough non-null LF/HF positions for a within-trial ranking, and within those LF/HF does not co-vary tightly enough with the click. LF/HF as a sole relevance proxy is rejected.
- **K29b succeeds.** Using LF/HF as a within-bucket tiebreaker on top of the 4-class outer label adds **+0.014 MRR@10 / +0.011 NDCG@10**. The 4-class hard label preserves the click ground truth; LF/HF densifies the within-bucket pairwise gradient (Deferred items with high LF/HF outrank Deferred items with low LF/HF; same for EvalRej / NotApprAbove). This is the "set of pairwise preferences" framing operationalized.

**Cumulative label-encoding trajectory on M3-no-position cursor features (typed cascade, 2,539 trials):**

| Step | Label | LambdaMART MRR@10 | Cumulative Δ over binary |
|---|---|---|---|
| K27.2 | binary click (NotApprBelow excluded) | 0.7409 | — |
| K27.3 | 4-class graded (NotApprBelow excluded) | 0.7713 | +0.0304 |
| **K29b** | **4-class × LF/HF composite** | **0.7854** | **+0.0445** |

Each refinement adds signal: the four-class behavioral structure recovers +0.030 MRR over binary, and pupillary LF/HF densifies the within-bucket order for another +0.014 on top of that. **Total label-encoding gain = +0.044 MRR@10 / +0.036 NDCG@10** over binary-click LambdaMART, holding features and folds constant.

**What this resolves about LF/HF as a relevance signal:**

- LF/HF is **not a stand-alone relevance ordering** (K29a, ΔMRR = −0.354) — it correlates with cognitive engagement, but engagement does not equal preference.
- LF/HF is **not a useful additional feature** at the per-(trial, pos) level (K28.1, ΔMRR = −0.001) — its information is largely redundant with cursor dwell.
- LF/HF **is** a useful within-bucket tiebreaker for LTR labels (K29b, ΔMRR = +0.014) — given the 4-class outer hard label, LF/HF adds finer gradient that the pairwise-loss LambdaMART can exploit.

This is consistent with the broader LF/HF story in the project memory: LF/HF is a trial-level / per-participant cognitive-load trait (load-decreases-with-position gradient, sat/opt orthogonality), not a within-trial preference discriminator. Used as a graded-label *refiner* (within-bucket order), it contributes; used as a primary feature or label, it does not.

**Caveats on K28/K29:**

- **LF/HF coverage is 28.7 %** (5,668 of 19,774 records). K29b handles this cleanly by treating null LF/HF as bucket-bottom; K29a is forced to subset to 514 trials with ≥3 non-null positions, which severely undertrains.
- **Same trial-pool as K27** (2,539 trials), apples-to-apples with K27.3.
- **No cross-trial pupil normalization yet.** LF/HF values are absolute (Butterworth IIR magnitudes) and not z-scored within participant. A normalized variant might improve K29b further; not run here.
- **K29b's pairwise-preference structure is implicit** — LightGBM's lambdarank objective converts the 10-grade composite label into pairwise preferences internally. An explicit pairwise-preference dataset (one row per (trial, i, j) pair with the relative judgment) would give the same gradient signal but is not necessary to validate the design.

**What's open after K29:**

- **Per-participant LF/HF normalization** — z-score within participant before assigning the within-bucket tiebreaker. Would test whether K29b's gain comes from absolute LF/HF magnitudes (partially redundant with M4 dwell) or from within-participant rank order (a cleaner pairwise signal).
- **Per-etype K29b decomposition** — does the LF/HF tiebreaker contribute differently for organic, dd_top, native_ad? Parallels `four_class_taxonomy_hybrid.py` for the four-class motor signature.
- **Combined LF/HF + RIPA2 tiebreaker** — RIPA2 is a per-fixation arousal metric (memory: "RIPA2 unique story"). A composite tiebreaker (high LF/HF AND high RIPA2 → strongest within-bucket preference) might add further gradient.


In [1]:
# ── Imports + paths ───────────────────────────────────────────────────
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ROOT = Path("/Users/andyed/Documents/dev/attentional-foraging")
sys.path.insert(0, str(ROOT / "notebooks-v2"))

LAB_RECORDS = ROOT / "AdSERP/data/cursor-approach-features.json"
REGRESSION_CACHE = ROOT / "scripts/output/approach_threshold_sensitivity/regression_labels_cache.json"
SERP_EMB_COMBINED = ROOT / "AdSERP/data/serp-embeddings.json"
SERP_EMB_SPLIT = ROOT / "AdSERP/data/serp-embeddings-split.json"
QUERY_EMB = ROOT / "AdSERP/data/query-embeddings.json"
print("imports ok")

imports ok


In [2]:
# ── Load AdSERP records, NB22 labels, embeddings ─────────────────────
lab_records = json.load(open(LAB_RECORDS))
regression_labels = np.array(json.load(open(REGRESSION_CACHE)), dtype=bool)
print(f"lab records: {len(lab_records):,}")
print(f"NB22 gaze_regression labels: {len(regression_labels):,}")

with open(SERP_EMB_COMBINED) as f:
    serp_combined = json.load(f)
with open(SERP_EMB_SPLIT) as f:
    serp_split = json.load(f)
with open(QUERY_EMB) as f:
    query_data = json.load(f)

print(f"combined embeddings: {len(serp_combined):,} trials")
print(f"split embeddings: {len(serp_split):,} trials")
print(f"query embeddings: {len(query_data):,} trials")

# Lookups keyed by (trial_id, position)
combined_emb = {}
combined_text = {}
for tid, rs in serp_combined.items():
    for r in rs:
        if "embedding" in r:
            combined_emb[(tid, r["position"])] = np.array(r["embedding"], dtype=np.float32)
            combined_text[(tid, r["position"])] = r.get("text", "")

title_emb = {}
snippet_emb = {}
title_text = {}
snippet_text = {}
for tid, rs in serp_split.items():
    for r in rs:
        key = (tid, r["position"])
        if "title_embedding" in r:
            title_emb[key] = np.array(r["title_embedding"], dtype=np.float32)
        if "snippet_embedding" in r:
            snippet_emb[key] = np.array(r["snippet_embedding"], dtype=np.float32)
        title_text[key] = r.get("title", "") or ""
        snippet_text[key] = r.get("snippet", "") or ""

query_emb_lookup = {}
query_text_lookup = {}
for tid, v in query_data.items():
    if isinstance(v, dict) and "embedding" in v:
        query_emb_lookup[tid] = np.array(v["embedding"], dtype=np.float32)
        query_text_lookup[tid] = v.get("query", "")

print(f"\nlookups built:")
print(f"  combined emb: {len(combined_emb):,}")
print(f"  title emb:    {len(title_emb):,}")
print(f"  snippet emb:  {len(snippet_emb):,}")
print(f"  query emb:    {len(query_emb_lookup):,}")

lab records: 13,419
NB22 gaze_regression labels: 13,419


combined embeddings: 2,776 trials
split embeddings: 2,776 trials
query embeddings: 2,776 trials



lookups built:
  combined emb: 27,520
  title emb:    27,520
  snippet emb:  27,520
  query emb:    2,776


In [3]:
# ── Apply the label scheme with exclusions ────────────────────────────
# For each trial, identify the clicked result position, then:
#   clicked            → 2
#   NB22 deferred      → 1
#   eval-rejected      → 0
#   not-approached ABOVE click → 0
#   not-approached BELOW click → EXCLUDED (structurally unseen)
#
# Also build the binary variant used by the 3-way comparison:
#   clicked → 1
#   everything else above click → 0
#   below click → excluded

# First: click position per trial
click_pos_by_trial = {}
for r in lab_records:
    if r.get("was_clicked"):
        click_pos_by_trial[r["trial_id"]] = r["position"]
print(f"trials with a click: {len(click_pos_by_trial):,}")

# Build per-record labels
graded_by_key = {}
binary_by_key = {}
excluded_count = 0
for i, r in enumerate(lab_records):
    tid = r["trial_id"]
    pos = r["position"]
    key = (tid, pos)
    click_pos = click_pos_by_trial.get(tid)
    if click_pos is None:
        continue  # no click in this trial — can't build the exclusion

    clicked = bool(r.get("was_clicked", False))
    min_dist = float(r.get("min_dist", 9999))
    approached = min_dist < 100
    deferred = bool(regression_labels[i]) if i < len(regression_labels) else False

    # Apply below-click exclusion for not-approached records only
    if not approached and not clicked and pos > click_pos:
        excluded_count += 1
        continue

    # Graded label
    if clicked:
        grade = 2
    elif approached and deferred:
        grade = 1
    else:
        grade = 0
    graded_by_key[key] = grade
    binary_by_key[key] = 1 if clicked else 0

print(f"graded-labeled records:   {len(graded_by_key):,}")
print(f"  grade 2 (clicked):      {sum(1 for g in graded_by_key.values() if g == 2):,}")
print(f"  grade 1 (deferred):     {sum(1 for g in graded_by_key.values() if g == 1):,}")
print(f"  grade 0 (rejected/nota): {sum(1 for g in graded_by_key.values() if g == 0):,}")
print(f"excluded not-approached-below-click records: {excluded_count:,}")

trials with a click: 2,228
graded-labeled records:   7,432
  grade 2 (clicked):      2,228
  grade 1 (deferred):     1,862
  grade 0 (rejected/nota): 3,342
excluded not-approached-below-click records: 5,480


In [4]:
# ── Compute the 5 features per (trial, position) ─────────────────────
# 1. lexical_overlap(query_tokens, title+snippet_tokens)
# 2. avg_term_frequency(query_tokens, title+snippet_tokens)
# 3. cos_sim(query, title)
# 4. cos_sim(query, snippet)
# 5. cos_sim(query, title+snippet)

STOPWORDS = set("a an and or but the is are was were be been being "
                "to of in on at for from by with as "
                "this that these those".split())

def tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", (text or "").lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def cos_sim(a, b):
    if a is None or b is None:
        return 0.0
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

FEATURE_NAMES = [
    "lex_overlap",
    "avg_tf",
    "cos_title",
    "cos_snippet",
    "cos_combined",
]

features_by_key = {}
skipped_missing_emb = 0
skipped_missing_query = 0
for key in graded_by_key:
    tid, pos = key
    q_emb = query_emb_lookup.get(tid)
    q_text = query_text_lookup.get(tid, "")
    if q_emb is None or not q_text:
        skipped_missing_query += 1
        continue
    t_emb = title_emb.get(key)
    s_emb = snippet_emb.get(key)
    c_emb = combined_emb.get(key)
    if t_emb is None or s_emb is None or c_emb is None:
        skipped_missing_emb += 1
        continue

    # Lexical features
    q_tokens = tokenize(q_text)
    r_text = (title_text.get(key, "") or "") + " " + (snippet_text.get(key, "") or "")
    r_tokens = tokenize(r_text)
    if not q_tokens:
        lex_overlap = 0.0
        avg_tf = 0.0
    else:
        r_counter = {}
        for t in r_tokens:
            r_counter[t] = r_counter.get(t, 0) + 1
        matches = [t for t in q_tokens if t in r_counter]
        lex_overlap = len(matches) / len(q_tokens)
        # Average TF across query tokens: absent tokens count as 0.
        avg_tf = float(np.mean([r_counter.get(t, 0) for t in q_tokens]))

    features_by_key[key] = [
        lex_overlap,
        avg_tf,
        cos_sim(q_emb, t_emb),
        cos_sim(q_emb, s_emb),
        cos_sim(q_emb, c_emb),
    ]

print(f"feature vectors computed: {len(features_by_key):,}")
print(f"  skipped (missing query): {skipped_missing_query}")
print(f"  skipped (missing embs):  {skipped_missing_emb}")

# Summary stats per feature
X_all = np.array(list(features_by_key.values()))
print(f"\nfeature matrix shape: {X_all.shape}")
for j, name in enumerate(FEATURE_NAMES):
    col = X_all[:, j]
    print(f"  {name:>13}  mean={col.mean():+.3f}  std={col.std():.3f}  min={col.min():+.3f}  max={col.max():+.3f}")

feature vectors computed: 7,428
  skipped (missing query): 0
  skipped (missing embs):  4

feature matrix shape: (7428, 5)
    lex_overlap  mean=+0.748  std=0.206  min=+0.000  max=+1.000
         avg_tf  mean=+1.938  std=0.872  min=+0.000  max=+7.857
      cos_title  mean=+0.823  std=0.136  min=+0.249  max=+0.991
    cos_snippet  mean=+0.769  std=0.086  min=+0.338  max=+0.941
   cos_combined  mean=+0.805  std=0.059  min=+0.461  max=+0.957


In [5]:
# ── Build the per-trial record structure for ranker training ─────────
# Each trial becomes a list of (features, graded_label, binary_label, position,
# was_clicked, participant) records. MRR is computed per trial.

trials_data = defaultdict(list)
for key, feats in features_by_key.items():
    tid, pos = key
    participant = tid.split("-")[0]
    trials_data[tid].append({
        "position": pos,
        "features": feats,
        "graded": graded_by_key[key],
        "binary": binary_by_key[key],
        "clicked": (graded_by_key[key] == 2),
        "participant": participant,
    })

# Trials with at least one clicked record AND at least one non-clicked
# in the labeled pool are the ones we can train on.
usable_trials = [
    tid for tid, rs in trials_data.items()
    if any(r["clicked"] for r in rs) and len(rs) >= 2
]
print(f"total trials: {len(trials_data):,}")
print(f"usable trials (has click + ≥2 labeled records): {len(usable_trials):,}")

# Distribution of records per trial
lengths = [len(trials_data[tid]) for tid in usable_trials]
print(f"records per usable trial: "
      f"mean={np.mean(lengths):.1f}, "
      f"median={int(np.median(lengths))}, "
      f"min={min(lengths)}, max={max(lengths)}")

total trials: 2,228
usable trials (has click + ≥2 labeled records): 1,919
records per usable trial: mean=3.7, median=3, min=2, max=10


In [6]:
# ── MRR helper: compute per-trial MRR@10 given a scoring function ───

def mrr_for_trial(trial_records, scoring_fn):
    """Given a list of records for a trial and a callable that takes a
    feature row and returns a score, return 1 / rank_of_clicked_result.
    Rank is 1-indexed. If the trial has no clicked record in the labeled
    pool (shouldn't happen for usable trials), returns 0."""
    if not trial_records:
        return 0.0
    scored = [(scoring_fn(r["features"]), r["clicked"]) for r in trial_records]
    # Sort by descending score; ties broken by insertion order
    scored.sort(key=lambda x: -x[0])
    for rank, (_, clicked) in enumerate(scored, 1):
        if clicked:
            return 1.0 / rank
    return 0.0

def mrr_original_order(trial_records):
    """MRR using the Google SERP ordering directly (position index).
    Note: this is MRR@N where N = labeled records, not always @10,
    because the exclusion rule drops below-click non-approached items."""
    if not trial_records:
        return 0.0
    sorted_by_pos = sorted(trial_records, key=lambda r: r["position"])
    for rank, r in enumerate(sorted_by_pos, 1):
        if r["clicked"]:
            return 1.0 / rank
    return 0.0

print("MRR helper defined")

MRR helper defined


In [7]:
# ── Leave-one-participant-out CV: train LR per fold, score held-out ──
# Two ranker variants: binary labels, graded labels. Same features, same splits.

def train_scoring_fn(train_records, label_key):
    """Return a callable that maps a feature row to a score.
    For graded labels we treat it as pointwise regression onto [0, 1, 2];
    for binary we fit logistic regression and use decision_function."""
    X = np.array([r["features"] for r in train_records])
    y = np.array([r[label_key] for r in train_records])
    if label_key == "binary":
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0)),
        ])
        pipe.fit(X, y)
        # decision_function returns log-odds; higher = more clickable
        return lambda feats: float(pipe.decision_function(np.asarray(feats).reshape(1, -1))[0])
    else:
        # Pointwise graded: standardize features and fit a linear regression on the
        # integer grades. Higher grade = more relevant. Sklearn LR would reduce to
        # a 3-class problem that loses ordinal structure; a linear regression on
        # the graded scores is the correct pointwise treatment for this comparison.
        from sklearn.linear_model import Ridge
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=1.0)),
        ])
        pipe.fit(X, y.astype(float))
        return lambda feats: float(pipe.predict(np.asarray(feats).reshape(1, -1))[0])

# Build participant → [trial_ids] index
participant_to_trials = defaultdict(list)
for tid in usable_trials:
    participant = trials_data[tid][0]["participant"]
    participant_to_trials[participant].append(tid)
participants = sorted(participant_to_trials.keys())
print(f"participants: {len(participants)}")

# LOSO-by-participant
per_trial_mrr_original = {}
per_trial_mrr_binary = {}
per_trial_mrr_graded = {}

for fold_i, held_out_p in enumerate(participants):
    held_out_tids = set(participant_to_trials[held_out_p])
    train_tids = [t for t in usable_trials if t not in held_out_tids]
    train_records_all = []
    for t in train_tids:
        train_records_all.extend(trials_data[t])

    score_binary = train_scoring_fn(train_records_all, "binary")
    score_graded = train_scoring_fn(train_records_all, "graded")

    for t in held_out_tids:
        recs = trials_data[t]
        per_trial_mrr_original[t] = mrr_original_order(recs)
        per_trial_mrr_binary[t] = mrr_for_trial(recs, score_binary)
        per_trial_mrr_graded[t] = mrr_for_trial(recs, score_graded)

    if fold_i % 10 == 0:
        print(f"  fold {fold_i + 1}/{len(participants)} done")

# Aggregate
aligned_tids = [t for t in usable_trials if t in per_trial_mrr_binary]
mrr_orig = np.array([per_trial_mrr_original[t] for t in aligned_tids])
mrr_bin = np.array([per_trial_mrr_binary[t] for t in aligned_tids])
mrr_grad = np.array([per_trial_mrr_graded[t] for t in aligned_tids])

print(f"\naligned held-out trials: {len(aligned_tids):,}")
print(f"\n[K1] MRR (original SERP): {mrr_orig.mean():.4f}")
print(f"[K2] MRR (binary labels): {mrr_bin.mean():.4f}")
print(f"[K3] MRR (graded labels): {mrr_grad.mean():.4f}")

print(f"\n[K4] graded − binary: {mrr_grad.mean() - mrr_bin.mean():+.4f}")
print(f"[K5] graded − original: {mrr_grad.mean() - mrr_orig.mean():+.4f}")

# Paired per-participant aggregation
by_participant = defaultdict(lambda: {"orig": [], "bin": [], "grad": []})
for t, mo, mb, mg in zip(aligned_tids, mrr_orig, mrr_bin, mrr_grad):
    p = trials_data[t][0]["participant"]
    by_participant[p]["orig"].append(mo)
    by_participant[p]["bin"].append(mb)
    by_participant[p]["grad"].append(mg)

per_p_orig = np.array([np.mean(v["orig"]) for v in by_participant.values()])
per_p_bin = np.array([np.mean(v["bin"]) for v in by_participant.values()])
per_p_grad = np.array([np.mean(v["grad"]) for v in by_participant.values()])

paired_bin_grad = per_p_grad - per_p_bin
paired_orig_grad = per_p_grad - per_p_orig

print(f"\nper-participant paired:")
print(f"  mean(graded) = {per_p_grad.mean():.4f} ± {per_p_grad.std(ddof=1):.4f} (47 participants)")
print(f"  mean(binary) = {per_p_bin.mean():.4f} ± {per_p_bin.std(ddof=1):.4f}")
print(f"  mean(orig)   = {per_p_orig.mean():.4f} ± {per_p_orig.std(ddof=1):.4f}")
print(f"\n  graded − binary paired delta: {paired_bin_grad.mean():+.4f} ± {paired_bin_grad.std(ddof=1):.4f}")
print(f"  graded ≥ binary in {int((paired_bin_grad >= 0).sum())}/{len(paired_bin_grad)} participants")

# Wilcoxon signed-rank (non-parametric paired test)
try:
    w_bin = wilcoxon(per_p_grad, per_p_bin, alternative="greater")
    print(f"\n  Wilcoxon (graded > binary): W = {w_bin.statistic:.2f}, p = {w_bin.pvalue:.4f}")
except Exception as e:
    print(f"  Wilcoxon failed: {e}")
try:
    w_orig = wilcoxon(per_p_grad, per_p_orig, alternative="greater")
    print(f"  Wilcoxon (graded > original): W = {w_orig.statistic:.2f}, p = {w_orig.pvalue:.4f}")
except Exception as e:
    print(f"  Wilcoxon (graded vs orig) failed: {e}")

participants: 47
  fold 1/47 done


  fold 11/47 done


  fold 21/47 done


  fold 31/47 done


  fold 41/47 done

aligned held-out trials: 1,919

[K1] MRR (original SERP): 0.4114
[K2] MRR (binary labels): 0.6108
[K3] MRR (graded labels): 0.6152

[K4] graded − binary: +0.0045
[K5] graded − original: +0.2038

per-participant paired:
  mean(graded) = 0.6187 ± 0.0654 (47 participants)
  mean(binary) = 0.6141 ± 0.0601
  mean(orig)   = 0.4122 ± 0.0702

  graded − binary paired delta: +0.0046 ± 0.0209
  graded ≥ binary in 31/47 participants

  Wilcoxon (graded > binary): W = 720.50, p = 0.0246
  Wilcoxon (graded > original): W = 1128.00, p = 0.0000


> **Note (2026-04-19): Superseded by the extension below (cell 10 onward).** The "If K4 lands positive" promotion gate in this cell predates the full-SERP reanalysis; see the 2026-04-19 section for the current paper-side gate.

## Results summary

After executing all cells, transcribe the K1–K5 values into the Key Claims block at the top.

**Interpretation guide:**

- **K4 > 0 (graded beats binary) with signed-rank p < 0.05**: the graded relevance labels add downstream ranker-training information that binary click labels miss. Direct empirical validation of the graded-relevance reframe (CIKM §4.2, §4.3).
- **K4 ≈ 0 (paired delta within noise)**: graded and binary labels produce equivalent rankers on the same features. Reframe still defensible (graded is more informative theoretically) but the empirical gain is undetected at this sample size.
- **K4 < 0**: graded labels hurt. Would indicate the deferred=1 labeling is noisy enough to drag the ranker. Unlikely given K5 in NB25 showed 83% vs 45% deferred rate on rubbernecking split, but possible.
- **K5 > 0 (graded beats Google SERP order)**: the 5-feature behavioral + lexical ranker with 47-fold LOPO improves on the original ranking. Independent of the binary-vs-graded contrast, this tells us whether the behavioral signal adds value on top of whatever Google's ranker did.
- **K5 < 0 (graded loses to Google)**: Google's ranker is strictly better on this feature set. Not surprising — Google has orders of magnitude more features and training data. The contrast that matters is K4.

**If K4 lands positive**: goes into §4.3 of the CIKM paper as the direct empirical validation of the graded-relevance reframe. ~150 words, one table.

## 2026-04-19 extension — Full-SERP MRR × LambdaMART × M4 cursor features

**Motivation.** Renewed review of the 2026-04-15 null (`docs/null-findings/nb26-ltr-graded-vs-binary.md`): don't leave it as a null. The doc's own "What we'd need" list names three paths. We pursue paths (i) and (ii):

1. **Rung 1**: swap LR/Ridge for LambdaMART (LightGBM LGBMRanker), same 5 text features. Isolates the ranker-family effect.
2. **Rung 2**: LambdaMART on 5 text + 9 M4 cursor features + an approached-flag. Isolates the feature-addition effect on top of Rung 1.

Evaluation shifts to **full-SERP MRR** (score all 10 positions per held-out trial, not just the labeled subset). This is stricter than the null doc's "scorable positions" protocol — the click ranks farther from the top when non-labeled positions are all in the pool — but it is the honest ranker-evaluation metric.

**M4 leakage disclaimer.** The 9 M4 features (`min_dist`, `retreat_dist`, `dwell_in_proximity_ms`, ...) are aggregates over the cursor's full trajectory on each result, which for clicked results includes the movement *to* the click target. Clicked records therefore have dramatically smaller min_dist (~73 px median vs ~235 px for non-clicked) and longer dwell (~1760 ms vs ~194 ms). Any "Rung 2 vs Google" comparison is therefore **not a deployable-ranker claim** — it is a post-hoc demonstration. The defensible isolated contrast at Rung 2 is the **graded-vs-binary paired Δ**: same features, same LambdaMART, leakage cancels across the contrast.


In [8]:
# ── 2026-04-19 extension imports + M4 feature load ──
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from lightgbm import LGBMRanker

M4_FEATURE_NAMES = [
    "min_dist", "mean_dist", "final_dist", "retreat_dist",
    "dwell_in_proximity_ms", "mean_approach_velocity", "max_approach_velocity",
    "direction_changes", "frac_decreasing",
]

# Index M4 per (trial_id, position). Missing entries become NaN (LGBM native).
m4_by_key = {}
for r in lab_records:
    m4_by_key[(r["trial_id"], r["position"])] = [
        float(r.get(f)) if r.get(f) is not None else np.nan
        for f in M4_FEATURE_NAMES
    ]
print(f"M4 records keyed by (trial, position): {len(m4_by_key):,}")

# Leakage sanity check: clicked vs not-clicked feature medians
clicked_rows = [r for r in lab_records if r.get("was_clicked")]
nc_rows = [r for r in lab_records if not r.get("was_clicked")]
print("\nLeakage audit (click-predictive separation in M4):")
for f in ["min_dist", "final_dist", "dwell_in_proximity_ms"]:
    c_med = np.median([r[f] for r in clicked_rows if r.get(f) is not None])
    nc_med = np.median([r[f] for r in nc_rows if r.get(f) is not None])
    print(f"  {f:25s}  clicked median={c_med:8.2f}   not-clicked median={nc_med:8.2f}")


M4 records keyed by (trial, position): 13,419

Leakage audit (click-predictive separation in M4):
  min_dist                   clicked median=   73.14   not-clicked median=  235.31
  final_dist                 clicked median=  182.65   not-clicked median=  427.89
  dwell_in_proximity_ms      clicked median= 1759.50   not-clicked median=  194.00


In [9]:
# ── Build per-trial records covering ALL 10 positions (full SERP) ──
# Training labels respect the not-approached-below-click exclusion;
# inference scores every position.

click_pos_by_trial = {}  # tid -> click position
for r in lab_records:
    if r.get("was_clicked"):
        click_pos_by_trial[r["trial_id"]] = r["position"]

# For each (tid, pos), compute or locate features; attach labels and eligibility.
full_serp = {}  # tid -> list of 10 records
for tid in click_pos_by_trial:
    q_emb = query_emb_lookup.get(tid)
    q_text = query_text_lookup.get(tid, "")
    q_tokens = tokenize(q_text)
    records = []
    ok = True
    for pos in range(10):
        key = (tid, pos)
        t_emb = title_emb.get(key)
        s_emb = snippet_emb.get(key)
        c_emb = combined_emb.get(key)
        t_txt = title_text.get(key, "")
        s_txt = snippet_text.get(key, "")
        if c_emb is None and t_emb is None and s_emb is None:
            ok = False
            break
        # Text features (same definition as cell 4)
        r_tokens = tokenize((t_txt or "") + " " + (s_txt or ""))
        if not q_tokens:
            lex = 0.0
            avg_tf_ = 0.0
        else:
            counter = {}
            for tok in r_tokens:
                counter[tok] = counter.get(tok, 0) + 1
            matches = [tok for tok in q_tokens if tok in counter]
            lex = len(matches) / len(q_tokens)
            avg_tf_ = float(np.mean([counter.get(tok, 0) for tok in q_tokens]))
        text_feats = [lex, avg_tf_, cos_sim(q_emb, t_emb), cos_sim(q_emb, s_emb), cos_sim(q_emb, c_emb)]

        # M4 + approached flag
        m4 = m4_by_key.get(key)
        has_m4 = m4 is not None
        m4_feats = list(m4) if m4 is not None else [np.nan] * 9
        # approached flag: requires M4 min_dist < 100
        approached_flag = 0
        if has_m4 and not np.isnan(m4[0]) and m4[0] < 100:
            approached_flag = 1

        # Labels (only meaningful for training-eligible records)
        grade = graded_by_key.get(key)
        binary = binary_by_key.get(key)
        clicked = (grade == 2) if grade is not None else False
        # Training eligibility: exclude not-approached BELOW click position
        click_pos = click_pos_by_trial[tid]
        is_below_nota = (not has_m4 or (has_m4 and not np.isnan(m4[0]) and m4[0] >= 100)) and not clicked and pos > click_pos
        train_eligible = (grade is not None) and (not is_below_nota)

        records.append({
            "position": pos,
            "text_feats": text_feats,
            "m4_feats": m4_feats,
            "approached_flag": approached_flag,
            "clicked": clicked,
            "graded": grade,
            "binary": binary,
            "train_eligible": train_eligible,
            "participant": tid.split("-")[0],
        })
    if not ok or len(records) != 10:
        continue
    # At least one click and ≥2 train-eligible records
    if not any(r["clicked"] for r in records):
        continue
    if sum(1 for r in records if r["train_eligible"] and r["graded"] is not None) < 2:
        continue
    full_serp[tid] = records

print(f"full-SERP trials (10 positions each): {len(full_serp):,}")
lens_elig = [sum(1 for r in recs if r["train_eligible"]) for recs in full_serp.values()]
print(f"train-eligible records per trial: mean={np.mean(lens_elig):.1f}, median={int(np.median(lens_elig))}")


full-SERP trials (10 positions each): 1,826
train-eligible records per trial: mean=3.7, median=3


In [10]:
# ── Full-SERP LOPO helper + MRR + paired Wilcoxon ──

def _features_for_mode(records, mode):
    if mode == "text":
        return np.array([r["text_feats"] for r in records], dtype=float)
    if mode == "text_m4":
        return np.array([r["text_feats"] + [r["approached_flag"]] + list(r["m4_feats"])
                         for r in records], dtype=float)
    raise ValueError(mode)

def mrr_from_scores(records, scores):
    order = np.argsort(-np.asarray(scores), kind="stable")
    for rank, idx in enumerate(order, 1):
        if records[idx]["clicked"]:
            return 1.0 / rank
    return 0.0

def mrr_original_full(records):
    for rank, r in enumerate(records, 1):
        if r["clicked"]:
            return 1.0 / rank
    return 0.0

def run_lopo_full(usable, label_key, feature_mode, ranker_kind, seed=0):
    trial_ids = sorted(usable.keys())
    pid_by_t = {t: usable[t][0]["participant"] for t in trial_ids}
    pid_to_t = defaultdict(list)
    for t, p in pid_by_t.items():
        pid_to_t[p].append(t)
    participants = sorted(pid_to_t.keys())

    mrr_by_trial = {}
    for held in participants:
        held_set = set(pid_to_t[held])
        train_tids = [t for t in trial_ids if t not in held_set]

        X_list, y_list, group_list = [], [], []
        for tid in train_tids:
            recs = usable[tid]
            elig = [r for r in recs if r["train_eligible"] and r[label_key] is not None]
            if len(elig) < 2:
                continue
            X_list.append(_features_for_mode(elig, feature_mode))
            y_list.append(np.array([r[label_key] for r in elig], dtype=float))
            group_list.append(len(elig))
        X_train = np.vstack(X_list)
        y_train = np.concatenate(y_list)
        groups = np.array(group_list, dtype=int)

        if ranker_kind == "lr":
            model = Pipeline([("s", StandardScaler()), ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0))])
            model.fit(X_train, y_train.astype(int))
            sf = lambda X: model.decision_function(X)
        elif ranker_kind == "ridge":
            from sklearn.linear_model import Ridge
            model = Pipeline([("s", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
            model.fit(X_train, y_train)
            sf = lambda X: model.predict(X)
        elif ranker_kind == "lgbm_binary":
            model = LGBMRanker(objective="lambdarank", label_gain=[0, 1], n_estimators=200,
                               num_leaves=31, learning_rate=0.05, min_child_samples=10,
                               random_state=seed, verbose=-1)
            model.fit(X_train, y_train.astype(int), group=groups)
            sf = lambda X: model.predict(X)
        elif ranker_kind == "lgbm_graded":
            model = LGBMRanker(objective="lambdarank", label_gain=[0, 1, 2], n_estimators=200,
                               num_leaves=31, learning_rate=0.05, min_child_samples=10,
                               random_state=seed, verbose=-1)
            model.fit(X_train, y_train.astype(int), group=groups)
            sf = lambda X: model.predict(X)
        else:
            raise ValueError(ranker_kind)

        for tid in held_set:
            recs = usable[tid]
            X_test = _features_for_mode(recs, feature_mode)
            scores = sf(X_test)
            assert len(scores) == 10, f"{tid}: {len(scores)} scores, expected 10"
            mrr_by_trial[tid] = mrr_from_scores(recs, scores)

    per_p = defaultdict(list)
    for tid, m in mrr_by_trial.items():
        per_p[pid_by_t[tid]].append(m)
    return {
        "per_trial": mrr_by_trial,
        "per_participant_mean": {p: float(np.mean(v)) for p, v in per_p.items()},
        "concat_mrr": float(np.mean(list(mrr_by_trial.values()))),
    }

def seed_avg(usable, label_key, feature_mode, ranker_kind, seeds=(0, 1, 2)):
    runs = [run_lopo_full(usable, label_key, feature_mode, ranker_kind, seed=s) for s in seeds]
    tids = sorted(set().union(*[set(r["per_trial"].keys()) for r in runs]))
    pid_by_t = {t: usable[t][0]["participant"] for t in tids}
    per_trial = {t: float(np.mean([r["per_trial"][t] for r in runs])) for t in tids}
    per_p = defaultdict(list)
    for t, m in per_trial.items():
        per_p[pid_by_t[t]].append(m)
    return {
        "per_trial": per_trial,
        "per_participant_mean": {p: float(np.mean(v)) for p, v in per_p.items()},
        "concat_mrr": float(np.mean(list(per_trial.values()))),
    }

def paired_wilcoxon(res_a, res_b, alternative="greater"):
    pids = sorted(set(res_a["per_participant_mean"].keys()) & set(res_b["per_participant_mean"].keys()))
    a = np.array([res_a["per_participant_mean"][p] for p in pids])
    b = np.array([res_b["per_participant_mean"][p] for p in pids])
    delta = a - b
    try:
        w = wilcoxon(a, b, alternative=alternative)
        W, p = float(w.statistic), float(w.pvalue)
    except Exception:
        W, p = float("nan"), float("nan")
    return {
        "delta_mean": float(delta.mean()),
        "delta_sd": float(delta.std(ddof=1)) if len(delta) > 1 else 0.0,
        "a_ge_b": int((a >= b).sum()),
        "n": len(pids),
        "W": W,
        "p": p,
    }

print("helpers defined")


helpers defined


In [11]:
# ── Run rungs 0, 1, 2 ──

# Baseline: Google original SERP ordering
orig_per_trial = {tid: mrr_original_full(recs) for tid, recs in full_serp.items()}
orig_per_p = defaultdict(list)
for tid, m in orig_per_trial.items():
    orig_per_p[full_serp[tid][0]["participant"]].append(m)
orig_result = {
    "per_trial": orig_per_trial,
    "per_participant_mean": {p: float(np.mean(v)) for p, v in orig_per_p.items()},
    "concat_mrr": float(np.mean(list(orig_per_trial.values()))),
}
K6 = orig_result["concat_mrr"]
print(f"[K6] Baseline full-SERP MRR (Google original ordering): {K6:.4f}")

# Rung 0: LR + Ridge on 5 text features (regression reference)
print("\n── Rung 0: LR / Ridge on 5 text features ──")
r0_bin = run_lopo_full(full_serp, "binary", "text", "lr")
r0_grad = run_lopo_full(full_serp, "graded", "text", "ridge")
K7 = r0_bin["concat_mrr"]
K8 = r0_grad["concat_mrr"]
print(f"  [K7] LR binary : {K7:.4f}")
print(f"  [K8] Ridge graded: {K8:.4f}")

# Rung 1: LambdaMART on 5 text features (seed-averaged)
print("\n── Rung 1: LambdaMART on 5 text features (seeds 0,1,2) ──")
r1_bin = seed_avg(full_serp, "binary", "text", "lgbm_binary")
r1_grad = seed_avg(full_serp, "graded", "text", "lgbm_graded")
K9 = r1_bin["concat_mrr"]
K10 = r1_grad["concat_mrr"]
print(f"  [K9] LGBM binary : {K9:.4f}")
print(f"  [K10] LGBM graded: {K10:.4f}")

# Rung 2: LambdaMART on 5 text + 9 M4 + approached flag (seed-averaged)
print("\n── Rung 2: LambdaMART on 5 text + 9 M4 cursor features (seeds 0,1,2) ──")
print("  ⚠ M4 features include post-click cursor behavior; vs-Google comparison is post-hoc only.")
r2_bin = seed_avg(full_serp, "binary", "text_m4", "lgbm_binary")
r2_grad = seed_avg(full_serp, "graded", "text_m4", "lgbm_graded")
K11 = r2_bin["concat_mrr"]
K12 = r2_grad["concat_mrr"]
print(f"  [K11] LGBM binary : {K11:.4f}")
print(f"  [K12] LGBM graded: {K12:.4f}")


[K6] Baseline full-SERP MRR (Google original ordering): 0.4125

── Rung 0: LR / Ridge on 5 text features ──


  [K7] LR binary : 0.2878
  [K8] Ridge graded: 0.2922

── Rung 1: LambdaMART on 5 text features (seeds 0,1,2) ──


  [K9] LGBM binary : 0.2893
  [K10] LGBM graded: 0.2821

── Rung 2: LambdaMART on 5 text + 9 M4 cursor features (seeds 0,1,2) ──
  ⚠ M4 features include post-click cursor behavior; vs-Google comparison is post-hoc only.


  [K11] LGBM binary : 0.3723
  [K12] LGBM graded: 0.4326


In [12]:
# ── Paired Wilcoxon table + K-value synthesis ──
comparisons = [
    ("R0 Ridge grad > R0 LR bin",        r0_grad,  r0_bin),
    ("R0 Ridge grad > Original",         r0_grad,  orig_result),
    ("R1 LGBM bin > Original",           r1_bin,   orig_result),
    ("R1 LGBM grad > Original",          r1_grad,  orig_result),
    ("R1 LGBM grad > R1 LGBM bin",       r1_grad,  r1_bin),
    ("R1 LGBM grad > R0 Ridge grad",     r1_grad,  r0_grad),  # ranker-family isolated
    ("R2 LGBM bin > Original",           r2_bin,   orig_result),
    ("R2 LGBM grad > Original",          r2_grad,  orig_result),
    ("R2 LGBM grad > R2 LGBM bin",       r2_grad,  r2_bin),   # graded-vs-binary at rung 2 (clean contrast)
    ("R2 LGBM grad > R1 LGBM grad",      r2_grad,  r1_grad),  # feature-add isolated
]
out = {}
print(f"{'comparison':40s}  {'Δ':>8s} ± {'sd':>6s}  {'win/n':>6s}   {'p':>7s}")
print("-" * 72)
for name, a, b in comparisons:
    r = paired_wilcoxon(a, b)
    out[name] = r
    print(f"  {name:40s}  {r['delta_mean']:+.4f} ± {r['delta_sd']:.4f}   {r['a_ge_b']:2d}/{r['n']}   p = {r['p']:.4f}")

# Headline K-values
K13 = out["R1 LGBM grad > R1 LGBM bin"]
K14 = out["R2 LGBM grad > R2 LGBM bin"]
K15 = out["R1 LGBM grad > R0 Ridge grad"]
K16 = out["R2 LGBM grad > R1 LGBM grad"]

print("\n── Key Claims (transcribe into cell-0 table) ──")
print(f"K6  Google baseline full-SERP MRR                  : {K6:.4f}")
print(f"K7  Rung 0 LR binary  full-SERP MRR                : {K7:.4f}")
print(f"K8  Rung 0 Ridge graded full-SERP MRR              : {K8:.4f}")
print(f"K9  Rung 1 LGBM binary full-SERP MRR               : {K9:.4f}")
print(f"K10 Rung 1 LGBM graded full-SERP MRR               : {K10:.4f}")
print(f"K11 Rung 2 LGBM binary full-SERP MRR (leakage)     : {K11:.4f}")
print(f"K12 Rung 2 LGBM graded full-SERP MRR (leakage)     : {K12:.4f}")
print(f"K13 graded − binary @ Rung 1 : Δ = {K13['delta_mean']:+.4f}, p = {K13['p']:.4f} ({K13['a_ge_b']}/{K13['n']})")
print(f"K14 graded − binary @ Rung 2 : Δ = {K14['delta_mean']:+.4f}, p = {K14['p']:.4f} ({K14['a_ge_b']}/{K14['n']})")
print(f"K15 Rung 1 grad − Rung 0 grad (ranker family): Δ = {K15['delta_mean']:+.4f}, p = {K15['p']:.4f} ({K15['a_ge_b']}/{K15['n']})")
print(f"K16 Rung 2 grad − Rung 1 grad (feature add)  : Δ = {K16['delta_mean']:+.4f}, p = {K16['p']:.4f} ({K16['a_ge_b']}/{K16['n']})")


comparison                                       Δ ±     sd   win/n         p
------------------------------------------------------------------------
  R0 Ridge grad > R0 LR bin                 +0.0039 ± 0.0259   29/47   p = 0.0719
  R0 Ridge grad > Original                  -0.1223 ± 0.0874    3/47   p = 1.0000
  R1 LGBM bin > Original                    -0.1222 ± 0.0868    2/47   p = 1.0000
  R1 LGBM grad > Original                   -0.1264 ± 0.0778    1/47   p = 1.0000
  R1 LGBM grad > R1 LGBM bin                -0.0042 ± 0.0450   22/47   p = 0.7891
  R1 LGBM grad > R0 Ridge grad              -0.0041 ± 0.0537   23/47   p = 0.6645
  R2 LGBM bin > Original                    -0.0512 ± 0.1250   17/47   p = 0.9962
  R2 LGBM grad > Original                   +0.0079 ± 0.1354   21/47   p = 0.4687
  R2 LGBM grad > R2 LGBM bin                +0.0591 ± 0.0826   39/47   p = 0.0000
  R2 LGBM grad > R1 LGBM grad               +0.1343 ± 0.1458   39/47   p = 0.0000

── Key Claims (transcribe in

## Interpretation (2026-04-19 extension)

**Did "trying harder" break the null?**

- **Ranker family alone (Rung 1 vs Rung 0)**: K15 is the clean ranker-family test on the original 5-text-feature set. If p > 0.05, LambdaMART does not meaningfully outperform Ridge on these features — the null is reinforced on the ranker-family axis.
- **Feature addition (Rung 2 vs Rung 1)**: K16 is the feature-add test. A strongly-positive Δ here, with a caveat, is the expected outcome because M4 features encode click-indicative cursor behavior.
- **Graded vs binary contrast**: K13 (at Rung 1) is the null-doc replication — it should stay non-significant. K14 (at Rung 2) is the key extension: if positive and significant, it shows the graded-label reframe earns empirical MRR credit **when the ranker has enough feature signal to express it** (same features, same ranker, leakage cancels in the contrast).

**What to report.**

- Rung 1 numbers (K9, K10, K13, K15) go into the null-findings doc as-is: the stronger ranker family did not break the null.
- Rung 2 graded-vs-binary contrast (K14) is the ledgerable extension finding. Report it with the M4 leakage caveat intact.
- Rung 2 vs Google (K11, K12) are not deployable claims. Report as informational context, flagged.

**Caveats to quote verbatim when reporting.** K14 isolates label encoding *plus* any loss-structure interaction with the leaky features — not label encoding alone. K15 compares a seed-averaged LGBM to a deterministic Ridge, so it is conservatively biased toward finding no effect (the null is conservatively established, not overstated). A pre-click-truncated M4 variant would attribute the K14 lift cleanly; it does not yet exist.

**WILD gate.** ACD (Attentive Cursor Dataset) has one AOI per session and no SERP HTML, so a graded-label LTR replication is blocked; a binary ad-click LTR with session-level M4 features would only probe the ranker-family question. Given K15 indicates LambdaMART alone does not buy much on text features, there is no pending "if it works" signal pointing to a WILD replication today. Document that the LAB null-extension reinforces the original null on the ranker-family axis, and defer WILD until a cursor-only deferred proxy exists (blocked per attentional-foraging/CLAUDE.md).

**After execution**: transcribe K6–K16 into the Key Claims table at the top of this notebook, then run `python notebooks-v2/update_key_claims.py` to refresh `docs/notebook-key-claims.md`.
